## Setup & Authentication

 Install required libraries (kaggle, pandas)    

In [29]:
import pandas as pd
from kaggle.api.kaggle_api_extended import KaggleApi
import os
import sys
import sqlite3

Create connection for sqlite

In [30]:
with sqlite3.connect("supermarket_sales.db", timeout=30) as conn:
    conn.execute("PRAGMA journal_mode=WAL;")

 Authenticate with Kaggle API via kaggle.json  

In [31]:
api = KaggleApi()
api.authenticate()

In [51]:
# Project folder
folder_path = r'C:\Users\theol\Desktop\Supermarket_Project\Data'

# Medallion/Quarantine folders
bronze_path = os.path.join(folder_path, 'bronze')
silver_path = os.path.join(folder_path, 'silver')
gold_path   = os.path.join(folder_path, 'gold')
quarantine_path = os.path.join(folder_path, 'quarantine')

# Create folders if they don't exist (idempotent!)
for path in [bronze_path, silver_path, gold_path, quarantine_path]:
    os.makedirs(path, exist_ok=True)

print("Folders ready!")

Folders ready!


## Raw Ingestion

 Download dataset via Kaggle API


In [ ]:
## Pull files from the Kaggle Dataset
dataset_files = api.dataset_list_files('lovishbansal123/sales-of-a-supermarket').files

## Loop through Kaggle Dataset files and save them to the folder path with the file name.
for file in dataset_files:
    file_path = os.path.join(folder_path, file.name)
    
    if not os.path.exists(file_path):
        api.dataset_download_file(
            'lovishbansal123/sales-of-a-supermarket',
            file_name=file.name,
            path=folder_path
        )
        print(f"Downloaded: {file.name}")
    else:
        print(f"Already exists, skipping: {file.name}")

Already exists, skipping: supermarket_sales.csv


In [ ]:
## Create a table to log loaded files with load timestamp

conn.execute("""
    CREATE TABLE IF NOT EXISTS loaded_files (
        file_name TEXT PRIMARY KEY,
        load_timestamp TIMESTAMP
    )
""")
conn.commit()

loaded_files = pd.read_sql("SELECT file_name FROM loaded_files", conn)
loaded_files_set = set(loaded_files["file_name"])

In [50]:
from datetime import datetime


all_dataframes = []
raw_df = pd.DataFrame()

## Loop through folder path to find CSVs, checking to see if file has been previously loaded.
for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        
        # Skip if already loaded
        if file in loaded_files_set:
            print(f"Skipping already loaded file: {file}")
            continue
        
        print(f"Loading new file: {file}")
        
        # Load CSV if not previously loaded
        raw_df = pd.read_csv(file_path)
        all_dataframes.append(raw_df)
        all_dataframes.to_parquet(os.path.join(bronze_path, "bronze_sales.parquet"), engine='fastparquet', index=False)
        
        # Mark as loaded
        conn.execute("INSERT OR IGNORE INTO loaded_files (file_name, load_timestamp) VALUES (?, ?)", (file,datetime.now().isoformat()))

## Checks to see if all_dataframes is empty, if so it prints that there are no new files. If a file or multiple files are found they are combined into raw_df. 
if not all_dataframes:
    print("No new files found.")
else:
    raw_df = pd.concat(all_dataframes, ignore_index=True)



No new files found.


## Data Quality & Quarantine

In [ ]:
## Function to normalize naming for columns to lower snakecase.
def column_renaming(df):
    new_columns = []
    for col in df.columns:
        if isinstance(col, str):
            new_col = col.strip().lower().replace(' ', '_')
        else:
            new_col = col
        new_columns.append(new_col)
    df.columns = new_columns
    return df

In [ ]:
# Define columns to check for data quality issues
null_check_columns = ['Invoice ID', 'Customer type', 'Gender', 'Total', 'Date']
dupe_check_columns = ["Invoice ID"]
numeric_check_columns = ["Total", "Unit price", "Quantity"]

# Tracking records that fail the null, dupe, and negative number checks.
quarantine = []

# Check for Nulls in null_check_columns
if not raw_df.empty:
    for col in null_check_columns:
         null_check = raw_df[col].isnull()
         df_null = raw_df[null_check].copy()
         df_null["quarantine_reason"] = f"Null value in column: {col}"
         quarantine.append(df_null)

    ## Check for Duplicates based on Invoice ID
    dupes = raw_df.duplicated(subset=dupe_check_columns, keep='first')
    df_dupes = raw_df[dupes].copy()
    df_dupes["quarantine_reason"] = f"Duplicate record based on Invoice ID"
    quarantine.append(df_dupes)

    ## Check for Negative values in numeric_check_columns
    for col in numeric_check_columns:
         neg_check = raw_df[col] < 0
         df_neg = raw_df[neg_check].copy()
         df_neg["quarantine_reason"] = f"Negative value found in {col}]"
         quarantine.append(df_neg)

    ## Filtering quarantine records to list only non duplicated records 
    ## Reasoning - duplicated records share an invoice ID with a valid record
    ## then pulls invoice IDs for records that violate other data quality checks
    non_dupe_quarantine = [q for q in quarantine if not q.empty and 'Duplicate' not in q['quarantine_reason'].values[0]] if quarantine else []
    bad_ids = pd.concat(non_dupe_quarantine)['Invoice ID'].unique() if non_dupe_quarantine else []

    ## Filter clean records on invoice ID
    df_clean = raw_df[~raw_df['Invoice ID'].isin(bad_ids)]
    df_clean = column_renaming(df_clean)
    df_clean.to_parquet(os.path.join(silver_path, "silver_sales.parquet"), engine='fastparquet')

    ## Combine quarantine records into one dataframe and normalizes naming
    df_quarantine = pd.concat(quarantine).drop_duplicates(subset=['Invoice ID'])
    df_quarantine = column_renaming(df_quarantine)

    print(f"Passed to silver: {len(df_clean)}")
    print(f"Quarantined - duplicates: {len(df_dupes)}")
    print(f"Total quarantined: {len(df_quarantine)}")

# Save quarantine
## Writes to Quarantine Parquet
    if not df_quarantine.empty:
        df_quarantine.to_parquet(os.path.join(quarantine_path, 'quarantine_sales.parquet'), engine='fastparquet')
        print("Quarantine file saved!")
else:
     df_clean = column_renaming(raw_df)
     print("No new records to add.")

No new records to add.


## Dimensions


 Create Customer dimension

In [ ]:
## Checks to see if df_clean has data
if not df_clean.empty:
    
    new_customer = df_clean[["customer_type", "gender"]].drop_duplicates().reset_index(drop=True)
    table_exists = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name='dim_customer';"
    ).fetchone()
    new_customer_df = pd.DataFrame()

    ##Checks to see if the dim_customer table exists
    if table_exists:
        existing_customer = pd.read_sql("SELECT * FROM dim_customer", conn)
        comparison_df = new_customer.merge(existing_customer[["customer_type", "gender"]],
                                      on= ["customer_type", "gender"],
                                      how="left", indicator=True)
        new_customer_df = comparison_df[comparison_df["_merge"]=="left_only"].drop(columns=["_merge"])
        ## Incremental load - if new_customer_df has data, records are written to the dim_customer table
        if not new_customer_df.empty:
            max_id = existing_customer["cust_id"].max()
            new_customer_df["cust_id"] = range(max_id +1, max_id + 1+ len(new_customer_df))
            new_customer_df.to_sql(
                "dim_customer", 
                conn, 
                if_exists="append", 
                index=False
                )
            new_customer_df.to_parquet(os.path.join(gold_path, 'dim_customer.parquet'), engine='fastparquet')
            silver_customer_df = pd.concat([existing_customer, new_customer_df], ignore_index=True)
        ## if new_customer_df is empty, prints that there are no new customers
        else:
            silver_customer_df = existing_customer
            print("No new customers.")
    ## Initial load - if table doesn't exist, table is created and new records are written
    else:
        new_customer["cust_id"] = new_customer.index + 1
        silver_customer_df = new_customer[["cust_id", "customer_type", "gender"]]
        ## Write to SQLite Database
        silver_customer_df.to_sql(
            "dim_customer", 
            conn, 
            if_exists="append", 
            index=False
            )
        ## Write to Gold Parquet
        silver_customer_df.to_parquet(os.path.join(gold_path, 'dim_customer.parquet'), engine='fastparquet')
    ## Prints count of new records added and total number of records
    if table_exists:
        new_record_count = len(new_customer_df) 
    else:
        new_record_count = len(new_customer)
    
    print(f"New records added: {new_record_count}")
    print(f"Total records: {len(silver_customer_df)}")
## If df_clean is empty, prints that there are no new records to process
else:
    print("No new records to process")


No new records to process



 Create Store dimension

In [ ]:
## Checks to see if df_clean has data
if not df_clean.empty:

    new_store = df_clean[["branch", "city"]].drop_duplicates().reset_index(drop=True)

    table_exists = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name='dim_store';"
    ).fetchone()

    ## Checks to see if the dim_store table exists
    if table_exists:
        existing_store = pd.read_sql("SELECT * FROM dim_store", conn)

        new_store_df = new_store.merge(existing_store[["branch", "city"]],
                                        on=["branch", "city"],
                                        how="left", indicator=True
                                        )
    
        new_store_df = new_store_df[new_store_df["_merge"]== "left_only"].drop(columns=["_merge"])

        ## Incremental load - if new_store_df has data, records are written to the dim_store table
        if not new_store_df.empty:
            max_id = existing_store["store_id"].max()
            new_store_df["store_id"] = range(max_id +1, max_id + 1 + len(new_store_df))

            new_store_df = new_store_df[["store_id", "branch", "city"]]
        
            new_store_df.to_sql(
                "dim_store",
                conn,
                if_exists="append",
                index=False
            )
            new_store_df.to_parquet(os.path.join(gold_path, 'dim_store.parquet'), engine='fastparquet')

            print(f"{len(new_store_df)} new records added.")
        ## If new_store_df is empty, prints that there are no new stores.
        else:
            silver_store_df = existing_store
            print("No new stores.")
    ## Initial load - if table doesn't exist, table is created and new records are written
    else:
        new_store["store_id"] = new_store.index + 1
        ## Write to SQLite Database
        new_store.to_sql(
            "dim_store",
            conn,
            if_exists="append",
            index=False
        )
        ## Write to Gold Parquet
        new_store.to_parquet(os.path.join(gold_path, 'dim_store.parquet'), engine='fastparquet')
        print(f"Initial load - {len(new_store)} records.")


else:
    print("No new records to process")


No new records to process


Create Product Dimension

In [ ]:
## Checks to see if df_clean has data in it
if not df_clean.empty:
    new_product = df_clean[["product_line"]].drop_duplicates().reset_index(drop=True)
    table_exists = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name='dim_product';"
    ).fetchone()
    new_product_df = pd.DataFrame()
    ## Checks to see if the dim_produt table exists 
    if table_exists:
        existing_product = pd.read_sql("SELECT * FROM dim_product", conn)
        comparison_df = new_product.merge(existing_product[["product_line"]],
                                        on=["product_line"],
                                        how="left", indicator=True)
        new_product_df = comparison_df[comparison_df["_merge"]== "left_only"].drop(columns=["_merge"])
        ## Checks to see if new_product_df is empty or it has data, if it has new data it assigns a cat_id and writes to the table
        if not new_product_df.empty:
            max_id = existing_product["cat_id"].max()
            new_product_df["cat_id"] = range(max_id + 1, max_id + 1 + len(new_product_df))
            new_product_df.to_sql(
                "dim_product", 
                conn, 
                if_exists="append", 
                index=False
                )
            new_product_df.to_parquet(os.path.join(gold_path, 'dim_product.parquet'), engine='fastparquet')
            silver_product_df = pd.concat([existing_product, new_product_df], ignore_index=True)
        ## If new_product_df is empty, prints that there are no new product lines to add
        else:
            silver_product_df = existing_product 
            print("No new products.")
    ## Initial load - If the dim_product table does not exist, loads the data into dim_product for the first time
    else:
        new_product["cat_id"] = new_product.index + 1
        silver_product_df = new_product[["cat_id", "product_line"]]
        ## Write to SQLite Database
        silver_product_df.to_sql(
            "dim_product",
            conn,
            if_exists="append",
            index=False
        )
        ## Write to Gold Parquet
        silver_product_df.to_parquet(os.path.join(gold_path, 'dim_product.parquet'), engine='fastparquet')
    ## Prints the number of new and total records for the table.
    if table_exists:
        new_record_count = len(new_product_df) 
    else:
        new_record_count = len(new_product)

    print(f"New records added: {new_record_count}")
    print(f"Total records: {len(silver_product_df)}")
## If the df_clean_df is empty, prints that there are no new records to process
else:
    print("No new records to process")



No new records to process


Create Date Dimension

In [ ]:
## Checks to see if df_clean has data or is empty.
if not df_clean.empty:

    new_date = df_clean[["date"]].drop_duplicates().reset_index(drop=True)
    new_date["date"] = pd.to_datetime(new_date["date"], errors="coerce")
    table_exists = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name='dim_date';"
    ).fetchone()
    ## If df_clean has data and dim_date exists, loads existing date data and compares with new data
    if table_exists:
        existing_date = pd.read_sql("SELECT * FROM dim_date", conn)
        existing_date["date"] = pd.to_datetime(existing_date["date"], errors="coerce")
        new_date_df = new_date[~new_date['date'].isin(existing_date['date'])]
        ## If no new data is found, prints that there are no new dates to add.
        if new_date_df.empty:
            print("No new dates to add.")
        ## If new data is found, creates date dimension with day, month, year, etc and writes to dim_date
        else:
            dim_date_df = new_date_df.copy()
            dim_date_df["date"] = dim_date_df["date"]
            dim_date_df['date_id'] = dim_date_df['date'].dt.strftime('%Y%m%d').astype(int)
            dim_date_df['day'] = dim_date_df['date'].dt.day
            dim_date_df['month'] = dim_date_df['date'].dt.month
            dim_date_df['year'] = dim_date_df['date'].dt.year
            dim_date_df['quarter'] = dim_date_df['date'].dt.quarter
            dim_date_df['day_of_week'] = dim_date_df['date'].dt.day_name()
            dim_date_df['month_name'] = dim_date_df['date'].dt.month_name()
            dim_date_df['week_of_year'] = dim_date_df['date'].dt.isocalendar().week
            dim_date_df["date"] = dim_date_df["date"].dt.date
        
            dim_date_df.to_sql(
                "dim_date",
                conn,
                if_exists="append",
                index=False
                )
            dim_date_df.to_parquet(os.path.join(gold_path, 'dim_date.parquet'), engine='fastparquet')
            print(f"{len(dim_date_df)} new records added.")
    ## Initial load - if table is not found, does initial load of date data including transformed day, month, year, etc columns
    else:
        dim_date_df = new_date.copy()
        dim_date_df['date_id'] = dim_date_df['date'].dt.strftime('%Y%m%d').astype(int)
        dim_date_df['day'] = dim_date_df['date'].dt.day
        dim_date_df['month'] = dim_date_df['date'].dt.month
        dim_date_df['year'] = dim_date_df['date'].dt.year
        dim_date_df['quarter'] = dim_date_df['date'].dt.quarter
        dim_date_df['day_of_week'] = dim_date_df['date'].dt.day_name()
        dim_date_df['month_name'] = dim_date_df['date'].dt.month_name()
        dim_date_df['week_of_year'] = dim_date_df['date'].dt.isocalendar().week
        
        ## Write to SQLite database
        dim_date_df.to_sql(
            "dim_date",
            conn,
            if_exists="append",
            index=False
        )
        ## Write to Gold Parquet
        dim_date_df.to_parquet(os.path.join(gold_path, 'dim_date.parquet'), engine='fastparquet')
    
        dim_date_df['date'] = dim_date_df['date'].dt.date

        print(f"Total records added: {len(dim_date_df)}")
## If df_clean is empty, prints that there are no new records to process.
else:
    print("No new records to process")


No new records to process


## Fact Table

Create Sales Fact Table

In [ ]:
## Checks to see if df_clean is empty, if not moves on to check for fact_sales table
if not df_clean.empty:

    new_sales = df_clean[
        ["invoice_id", "customer_type", "gender", "unit_price", "product_line", "branch", "city", "quantity", "tax_5%", "total", "date", 
         "time", "payment", "cogs", "gross_margin_percentage", "gross_income", "rating"]
         ].drop_duplicates().reset_index(drop=True)

    table_exists = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name='fact_sales';"
    ).fetchone()

    ## Checks to see if fact_sales exists
    if table_exists:
        existing_fact = pd.read_sql("SELECT * FROM fact_sales", conn)

        new_sales_df = new_sales[~new_sales["invoice_id"].isin(existing_fact["invoice_id"])]
        ## if new_sales)df is  not empty, pulls data from existing dimensions and joins to new_sales_df.
        ## Adds store_id, cust_id, date_id, and store_id to new_sales_df
        ## Writes to fact_sales
        if not new_sales_df.empty:
            dim_store_df = pd.read_sql("SELECT * FROM dim_store", conn)
            dim_customer_df = pd.read_sql("SELECT * FROM dim_customer", conn)
            dim_date_df = pd.read_sql("SELECT * FROM dim_date", conn)
            dim_product_df = pd.read_sql("SELECT * FROM dim_product", conn)
        
            new_sales_df = new_sales_df.merge(dim_store_df[['store_id','branch','city']],
                                      on=['branch','city'], how='left')
            new_sales_df = new_sales_df.merge(dim_customer_df[['cust_id','customer_type','gender']],
                                      on=['customer_type','gender'], how='left')
            new_sales_df['date'] = pd.to_datetime(new_sales_df['date'], errors='coerce')
            new_sales_df = new_sales_df.merge(dim_date_df[['date_id','date']],
                                      on='date', how='left')
            new_sales_df = new_sales_df.merge(dim_product_df[['cat_id','product_line']],
                                      on='product_line', how='left')
        
            fact_columns = ['invoice_id','store_id','cust_id','date_id','cat_id',
                    'unit_price','quantity','tax_5%','total','time','payment',
                    'cogs','gross_margin_percentage','gross_income','rating']
        
            new_sales_df = new_sales_df[fact_columns]
        
            new_sales_df.to_sql(
                "fact_sales",
                conn,
                if_exists="append",
                index=False
            )
            new_sales_df.to_parquet(os.path.join(gold_path, 'fact_sales.parquet'), engine='fastparquet')
            print(f"{len(new_sales_df)} new records added.")
        ## If new_sales_df is empty, prints that there are no new records to add.
        else:
            silver_sales_df = existing_fact
            print("No new invoices.")
    ## Initial load - if table Fact_sales does not exist, does initial load
    ## Joins existing dimensions and adds cust_id, Store_id, date_id, and cat_id to fact table
    else:
        dim_store_df = pd.read_sql("SELECT * FROM dim_store", conn)
        dim_customer_df = pd.read_sql("SELECT * FROM dim_customer", conn)
        dim_date_df = pd.read_sql("SELECT * FROM dim_date", conn)
        dim_product_df = pd.read_sql("SELECT * FROM dim_product", conn)

        new_sales['date'] = pd.to_datetime(new_sales['date'], errors='coerce')
        dim_date_df["date"] = pd.to_datetime(dim_date_df["date"], errors='coerce')
        new_sales = new_sales.merge(dim_store_df[['store_id','branch','city']], on=['branch','city'], how='left')
        new_sales = new_sales.merge(dim_customer_df[['cust_id','customer_type','gender']], on=['customer_type','gender'], how='left')
        new_sales = new_sales.merge(dim_date_df[['date_id','date']], on='date', how='left')
        new_sales = new_sales.merge(dim_product_df[['cat_id','product_line']], on='product_line', how='left')

        fact_columns = ['invoice_id','store_id','cust_id','date_id','cat_id',
                    'unit_price','quantity','tax_5%','total','time','payment',
                    'cogs','gross_margin_percentage','gross_income','rating']
        ## Write to SQLite database
        new_sales[fact_columns].to_sql(
            "fact_sales", 
            conn, 
            if_exists="append", 
            index=False
            )
        ## Write to Gold Parquet
        new_sales[fact_columns].to_parquet(os.path.join(gold_path, 'fact_sales.parquet'), engine='fastparquet')
        print(f"Initial load - {len(new_sales)} records.")

## If df_clean is empty, prints that there are no new records to process
else:
    print("No new records to process")

No new records to process


In [46]:
## Commits all changes and closes connection
conn.commit()
conn.close()

## GOLD LAYER

In [45]:
with sqlite3.connect("supermarket_sales.db") as conn:
    gold_revenue_by_branch = pd.read_sql("""
        SELECT s.branch, s.city, SUM(f.total) as total_revenue
        FROM fact_sales f
        JOIN dim_store s ON f.store_id = s.store_id
        GROUP BY s.branch, s.city
        ORDER BY total_revenue DESC
    """, conn)

print(gold_revenue_by_branch)

  branch       city  total_revenue
0      C  Naypyitaw    110568.7065
1      A     Yangon    106200.3705
2      B   Mandalay    106197.6720
